<a href="https://colab.research.google.com/github/chaitanyatalakeri27-png/Data_Science_Lab/blob/main/Exp7_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# OPTIMIZATION 1: Completely new dataset for totally different outputs
input_texts = [
    "what is your name",
    "where are you from",
    "how old are you",
    "what time is it",
    "goodbye my friend"
]

target_texts = [
    "i am a bot",
    "i live in the cloud",
    "i am brand new",
    "it is time to learn",
    "see you later"
]

# Tokenizer
tokenizer = Tokenizer()

tokenizer.fit_on_texts(input_texts + target_texts)

# Convert Text to Sequences
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

# Padding
max_len = max(
    max(len(seq) for seq in input_sequences),
    max(len(seq) for seq in target_sequences)
)

encoder_input_data = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    target_sequences,
    maxlen=max_len,
    padding='post'
)

# Vocabulary Size
vocab_size = len(tokenizer.word_index) + 1

# One Hot Encode Target
decoder_target_data = tf.keras.utils.to_categorical(
    decoder_input_data,
    num_classes=vocab_size
)

# OPTIMIZATION 2: Reduced latent_dim to 32 for faster execution
latent_dim = 32

# Encoder
encoder_inputs = Input(shape=(max_len,))

encoder_embedding = tf.keras.layers.Embedding(
    vocab_size,
    latent_dim
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len,))

decoder_embedding = tf.keras.layers.Embedding(
    vocab_size,
    latent_dim
)(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    vocab_size,
    activation='softmax'
)

decoder_outputs = decoder_dense(decoder_outputs)

# Seq2Seq Model
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

# Model Summary
model.summary()

# OPTIMIZATION 3: Swapped to RMSprop optimizer
model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Model
# OPTIMIZATION 4: Slashed epochs to 50 and increased batch_size to 5
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=5,
    epochs=50,
    verbose=1
)

# Evaluate Model
loss, accuracy = model.evaluate(
    [encoder_input_data, decoder_input_data],
    decoder_target_data
)

print("\nModel Loss :", loss)
print("Model Accuracy :", accuracy)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 5, 32)     │        960 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 5, 32)     │        960 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 32),      │      8,320 │ embedding[0][0]   │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 5, 32),   │      8,320 │ embedding_1[0][0… │
│                     │ (None, 32),       │            │ lstm[0][1],       │
│                     │ (None, 32)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 5, 30)     │        990 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,550 (76.37 KB)

 Trainable params: 19,550 (76.37 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0400 - loss: 3.4006
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2400 - loss: 3.3900
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2800 - loss: 3.3819
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3200 - loss: 3.3747
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3600 - loss: 3.3679
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3200 - loss: 3.3613
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2800 - loss: 3.3547
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.2800 - loss: 3.3479
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2800 - loss: 3.3410
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2800 - loss: 3.3339
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2800 - loss: 3.3264
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2800 - loss: 3.3186
Epo